# RetinaVessel AI — validation robustness and efficiency v3

This notebook evaluates the three verified seed-42 checkpoints on the frozen 89-image FIVES development-validation partition under the nine predeclared clean/degraded conditions. It records pixel metrics, clDice, thin-centerline recall, skeleton fragmentation, native-resolution latency, peak CUDA memory and image-bootstrap intervals.

**Protocol boundary:** attach the FIVES Kaggle dataset and either the v2 notebook output or its `retinavessel_v2_seed42.zip` artifact. This notebook must not inspect or score the official FIVES test images. Enable a GPU accelerator before running.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "f17833fdb25c9232e1df9cd60b0b5eb3ede048d6"
REPO = Path("/kaggle/working/retinavessel-ai-v3-code")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned evaluation code:", actual_commit)

Cloning into '/kaggle/working/retinavessel-ai-v3-code'...
HEAD is now at f17833f Add validation robustness evaluation framework


Pinned evaluation code: f17833fdb25c9232e1df9cd60b0b5eb3ede048d6


In [2]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest, samples_from_frozen_split

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v3"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
EXPECTED_SPLIT_SHA256 = "aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87"
assert hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = json.loads(SPLIT_MANIFEST.read_text())
_, validation = samples_from_frozen_split(load_manifest(FIVES_ROOT, "train"), split)
assert len(validation) == 89
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Frozen development validation images:", len(validation))
print("Official test images opened or evaluated: 0")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

PyTorch: 2.11.0+cu130
CUDA device: Tesla T4
Frozen development validation images: 89
Official test images opened or evaluated: 0
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], returncode=0)

In [3]:
INPUT_ROOT = Path("/kaggle/input")
EXTRACTED = Path("/kaggle/working/v2_verified_artifacts")
archives = sorted(INPUT_ROOT.rglob("retinavessel_v2_seed42.zip"))
if archives and not EXTRACTED.exists():
    EXTRACTED.mkdir(parents=True)
    shutil.unpack_archive(archives[0], EXTRACTED)
SEARCH_ROOTS = [INPUT_ROOT] + ([EXTRACTED] if EXTRACTED.exists() else [])

EXPECTED_CHECKPOINTS = {
    "unet_seed42": ("unet_seed42_es", "202a2d387663fd6e2e8db3463e79ce64fb72e278bc6b4655475a893b3f3162ad"),
    "lite_unet_seed42": ("lite_unet_seed42_es", "e1022147a4ed0f44c918ee694c53628687a3dbe687370ff79d30cf6908f63abb"),
    "lite_robust_seed42": ("lite_robust_seed42_es", "b48e12f1e17c7434f14b2adf51b2aac976b7a07b007049d7463eac34fa567395"),
}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verified_checkpoint(run_directory, expected_sha256):
    candidates = []
    for root in SEARCH_ROOTS:
        candidates.extend(path for path in root.rglob("best.pt") if path.parent.name == run_directory)
    matches = [path for path in candidates if file_sha256(path) == expected_sha256]
    if not matches:
        raise FileNotFoundError(f"Attach verified v2 artifacts; missing {run_directory}/best.pt")
    return sorted(matches, key=lambda path: (len(path.parts), str(path)))[0]

CHECKPOINTS = {
    model_id: verified_checkpoint(run_directory, expected_sha256)
    for model_id, (run_directory, expected_sha256) in EXPECTED_CHECKPOINTS.items()
}
for model_id, checkpoint in CHECKPOINTS.items():
    print(model_id, checkpoint, file_sha256(checkpoint))

unet_seed42 /kaggle/working/v2_verified_artifacts/outputs/unet_seed42_es/best.pt 202a2d387663fd6e2e8db3463e79ce64fb72e278bc6b4655475a893b3f3162ad
lite_unet_seed42 /kaggle/working/v2_verified_artifacts/outputs/lite_unet_seed42_es/best.pt e1022147a4ed0f44c918ee694c53628687a3dbe687370ff79d30cf6908f63abb
lite_robust_seed42 /kaggle/working/v2_verified_artifacts/outputs/lite_robust_seed42_es/best.pt b48e12f1e17c7434f14b2adf51b2aac976b7a07b007049d7463eac34fa567395


In [4]:
manifest_candidates = []
for root in SEARCH_ROOTS:
    manifest_candidates.extend(root.rglob("run_manifest.json"))
verified_manifests = []
for path in manifest_candidates:
    payload = json.loads(path.read_text())
    if (payload.get("split_manifest_sha256") == EXPECTED_SPLIT_SHA256
            and payload.get("official_test_evaluated") is False
            and len(payload.get("results", [])) == 3):
        verified_manifests.append((path, payload))
assert verified_manifests, "No valid v2 run_manifest.json found in attached artifacts"
V2_MANIFEST_PATH, V2_MANIFEST = sorted(verified_manifests, key=lambda item: str(item[0]))[0]
assert V2_MANIFEST["training_code_commit"] == "0c40aa665be3a247372062eaa46a33f578cf7b0f"
assert V2_MANIFEST["maximum_epochs"] == 30
assert V2_MANIFEST["early_stopping_patience"] == 5
print("Verified v2 run manifest:", V2_MANIFEST_PATH)

Verified v2 run manifest: /kaggle/input/notebooks/simonkosgei/retinavessel-ai-v2/retinavessel-ai-v2-code/outputs/v2_summary/run_manifest.json


In [5]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), str(REPO / "backend")])
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

..............                                                           [100%]
14 passed in 10.13s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## Full validation-only run

This executes 3 models × 9 conditions × 89 images = 2,403 native-resolution inferences. Do not add `--validation-limit`; that option is reserved for debugging. The exact severity and metric definitions are committed in `docs/v3_validation_protocol.md`.

In [6]:
OUTPUT = REPO / "outputs" / "validation_v3_seed42"
command = [
    sys.executable, "-m", "retinavessel.validation",
    "--data", str(DATA_MOUNT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--expected-split-sha256", EXPECTED_SPLIT_SHA256,
    "--model", f"unet_seed42={CHECKPOINTS['unet_seed42']}",
    "--model", f"lite_unet_seed42={CHECKPOINTS['lite_unet_seed42']}",
    "--model", f"lite_robust_seed42={CHECKPOINTS['lite_robust_seed42']}",
    "--topology",
    "--device", "cuda",
    "--warmup-runs", "2",
    "--degradation-seed", "2026",
    "--bootstrap-seed", "2026",
    "--bootstrap-repetitions", "2000",
    "--out", str(OUTPUT),
]
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
print("Running full v3 evaluation; this is expected to take substantial GPU time.")
subprocess.run(command, cwd=REPO, env=evaluation_environment, check=True)

Running full v3 evaluation; this is expected to take substantial GPU time.
model=unet_seed42, condition=clean, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed42, condition=blur_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed42, condition=blur_severe, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed42, condition=contrast_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed42, condition=contrast_severe, images=89
  processed 10/89
 

CompletedProcess(args=['/usr/bin/python3', '-m', 'retinavessel.validation', '--data', '/kaggle/input/datasets/simonkosgei/fives-retinal-dataset', '--split-manifest', '/kaggle/working/retinavessel-ai-v3-code/splits/fives_seed42.json', '--expected-split-sha256', 'aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87', '--model', 'unet_seed42=/kaggle/working/v2_verified_artifacts/outputs/unet_seed42_es/best.pt', '--model', 'lite_unet_seed42=/kaggle/working/v2_verified_artifacts/outputs/lite_unet_seed42_es/best.pt', '--model', 'lite_robust_seed42=/kaggle/working/v2_verified_artifacts/outputs/lite_robust_seed42_es/best.pt', '--topology', '--device', 'cuda', '--warmup-runs', '2', '--degradation-seed', '2026', '--bootstrap-seed', '2026', '--bootstrap-repetitions', '2000', '--out', '/kaggle/working/retinavessel-ai-v3-code/outputs/validation_v3_seed42'], returncode=0)

In [7]:
import pandas as pd

report = json.loads((OUTPUT / "validation_report.json").read_text())
per_image = pd.read_csv(OUTPUT / "validation_per_image.csv")
summary = pd.read_csv(OUTPUT / "validation_summary.csv")
drops = pd.read_csv(OUTPUT / "paired_degradation_drops.csv")
assert report["official_test_evaluated"] is False
assert report["validation_image_count"] == 89
assert report["debug_validation_limit"] is None
assert len(per_image) == 3 * 9 * 89
assert len(summary) == 3 * 9
assert set(per_image["image_id"]) == set(split["validation"])
display(summary[["model_id", "condition", "mean_dice", "mean_cldice", "mean_thin_centerline_recall", "median_latency_ms", "maximum_peak_accelerator_bytes"]])
display(drops)
print("Official test evaluated:", report["official_test_evaluated"])

,model_id,condition,mean_dice,mean_cldice,mean_thin_centerline_recall,median_latency_ms,maximum_peak_accelerator_bytes
0,unet_seed42,clean,0.823469,0.832439,0.215944,815.352787,85031936
1,unet_seed42,blur_mild,0.823144,0.832092,0.215703,827.872985,85031936
2,unet_seed42,blur_severe,0.815285,0.823640,0.196827,825.593333,85031936
3,unet_seed42,contrast_mild,0.785460,0.772736,0.128382,825.220180,85031936
4,unet_seed42,contrast_severe,0.705868,0.640779,0.071660,825.946093,85031936
5,unet_seed42,illumination_mild,0.801742,0.800893,0.158023,829.645395,85031936
6,unet_seed42,illumination_severe,0.741520,0.722609,0.091584,824.222925,85031936
7,unet_seed42,noise_mild,0.802149,0.807591,0.158289,832.378510,85031936
8,unet_seed42,noise_severe,0.737631,0.734861,0.090091,828.843810,85031936
9,lite_unet_seed42,clean,0.826155,0.837423,0.309949,445.820347,97457664


,model_id,condition,category,severity,image_count,mean_dice_drop,dice_drop_ci95_lower,dice_drop_ci95_upper,mean_cldice_drop,cldice_drop_ci95_lower,cldice_drop_ci95_upper,mean_thin_centerline_recall_drop,thin_centerline_recall_drop_ci95_lower,thin_centerline_recall_drop_ci95_upper
0,lite_robust_seed42,blur_mild,blur,mild,89,0.000036,-0.000141,0.000175,0.000034,-0.000230,0.000277,-0.000181,-0.001213,0.000807
1,lite_robust_seed42,blur_severe,blur,severe,89,0.003935,0.002653,0.005082,0.003192,0.001679,0.004575,0.019728,0.014791,0.024784
2,lite_robust_seed42,contrast_mild,contrast,mild,89,0.024745,0.018331,0.031312,0.027475,0.020881,0.034336,0.097968,0.087014,0.108704
3,lite_robust_seed42,contrast_severe,contrast,severe,89,0.086398,0.071560,0.102993,0.104095,0.088543,0.119580,0.216618,0.196737,0.237183
4,lite_robust_seed42,illumination_mild,illumination,mild,89,0.019904,0.011798,0.028273,0.027151,0.017843,0.037038,0.093547,0.079081,0.107085
5,lite_robust_seed42,illumination_severe,illumination,severe,89,0.078010,0.059436,0.099278,0.102239,0.082524,0.123503,0.218417,0.194559,0.240250
6,lite_robust_seed42,noise_mild,noise,mild,89,0.020403,0.016642,0.024428,0.025130,0.021278,0.029282,0.100349,0.088027,0.112713
7,lite_robust_seed42,noise_severe,noise,severe,89,0.077152,0.066452,0.088767,0.093006,0.082506,0.104425,0.203373,0.181047,0.226101
8,lite_unet_seed42,blur_mild,blur,mild,89,0.000351,0.000204,0.000492,0.000189,-0.000019,0.000385,-0.000024,-0.001123,0.001044
9,lite_unet_seed42,blur_severe,blur,severe,89,0.006941,0.005376,0.008290,0.005425,0.004098,0.006676,0.013497,0.007560,0.019348


Official test evaluated: False


In [8]:
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v3_validation_seed42")
archive = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=OUTPUT.parent, base_dir=OUTPUT.name))
print(f"Artifact archive: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Save this notebook version, then download the archive before starting multi-seed replication.")

Artifact archive: /kaggle/working/retinavessel_v3_validation_seed42.zip (0.3 MiB)
Save this notebook version, then download the archive before starting multi-seed replication.


## Stop here

Do not run the official FIVES test evaluation from this notebook. Review the complete validation artifacts, then run the separately versioned additional-seed training stage. Freeze the winning model and final analysis policy before opening the official test partition once.